# Phase 1.6: Bootstrap Null Synthetic Calibration

Calibrate the markovianity test statistic T_obs against bootstrap null distributions using synthetic scenarios.

Scenarios:
- **order1_unconfounded**: Clean VAR(1) null (Markovian)
- **order1_with_latent_confounder**: VAR(1) + latent common driver (positive control)
- **order3_unconfounded**: VAR(3) unconfounded (misspecified null)

For each scenario:
- Run calibration with B=50 bootstrap replicates
- Compute critical values at [0.90, 0.95, 0.99] quantiles
- Verify p-value ∈ [0, 1] and critical values are ordered
- Export results and figures to outputs/calibration/synthetic/

In [ ]:
from __future__ import annotations

import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find project root")

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import (
    scenario_order1_unconfounded,
    scenario_order3_unconfounded,
    scenario_latent_common_driver,
)
from markovianity_diagnostic.experiments.bootstrap import bootstrap_global_test
from markovianity_diagnostic.experiments.adapters import METHODS

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'calibration' / 'synthetic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

B = 50
SEED = 42
CRITICAL_LEVELS = [0.90, 0.95, 0.99]
P_VALUES = [1, 2, 3, 4, 5]
BLOCK_LENGTH = 1
METHOD_NAME = 'gcstar_cgc'

SCENARIOS = [
    ('order1_unconfounded', scenario_order1_unconfounded, {}),
    ('order1_with_latent_confounder', scenario_latent_common_driver, {}),
    ('order3_unconfounded', scenario_order3_unconfounded, {}),
]

print(f"Output directory: {OUTPUT_DIR}")
print(f"Bootstrap replicates: {B}")
print(f"Critical levels: {CRITICAL_LEVELS}")
print(f"P-values: {P_VALUES}")

In [ ]:
if METHOD_NAME not in METHODS:
    raise ValueError(f"Method {METHOD_NAME} not found. Available: {sorted(METHODS.keys())}")

analyze_fn = METHODS[METHOD_NAME]
print(f"Loaded analyzer: {METHOD_NAME}")

results = {}

for scenario_name, scenario_fn, scenario_kwargs in SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Calibrating: {scenario_name}")
    print(f"{'='*60}")
    
    sample = scenario_fn(T=2000, d=10, seed=SEED)
    X = sample.X
    print(f"Synthetic data shape: {X.shape}")
    
    result = bootstrap_global_test(
        X,
        analyze_fn=analyze_fn,
        p_values=P_VALUES,
        p0=1,
        B=B,
        block_length=BLOCK_LENGTH,
        seed=SEED + 1000,
    )
    
    T_obs = result.get('T_obs', 0.0)
    T_boot = result.get('T_boot', [])
    D_obs = result.get('D_obs', {})
    D_boot = result.get('D_boot', {})
    
    critical_values = {}
    T_boot_array = np.asarray(T_boot)
    for level in CRITICAL_LEVELS:
        critical_values[level] = float(np.quantile(T_boot_array, level))
    
    p_value = float((1 + np.sum(T_boot_array >= T_obs)) / (B + 1))
    
    assert 0 <= p_value <= 1, f"p-value {p_value} out of [0, 1]"
    
    results[scenario_name] = {
        'T_obs': T_obs,
        'T_boot': T_boot,
        'D_obs': D_obs,
        'D_boot': D_boot,
        'critical_values': critical_values,
        'p_value': p_value,
        'metadata': sample.metadata,
    }
    
    print(f"T_obs: {T_obs:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Critical values: {critical_values}")
    print(f"✓ Validation passed")

In [ ]:
def _get_git_commit() -> str:
    try:
        import subprocess
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=5,
            cwd=PROJECT_ROOT,
        )
        return result.stdout.strip() if result.returncode == 0 else "unknown"
    except Exception:
        return "unknown"

bootstrap_results_json = {}
for scenario_name, data in results.items():
    bootstrap_results_json[scenario_name] = {
        'T_obs': float(data['T_obs']),
        'T_boot': [float(x) for x in data['T_boot']],
        'critical_values': data['critical_values'],
        'p_value': float(data['p_value']),
    }

bootstrap_results_path = OUTPUT_DIR / 'bootstrap_results.json'
with open(bootstrap_results_path, 'w') as f:
    json.dump(bootstrap_results_json, f, indent=2)

print(f"Exported: {bootstrap_results_path}")

In [ ]:
summary_data = []
for scenario_name, data in results.items():
    row = {
        'scenario': scenario_name,
        'T_obs': float(data['T_obs']),
        'p_value': float(data['p_value']),
        'critical_90': data['critical_values'][0.90],
        'critical_95': data['critical_values'][0.95],
        'critical_99': data['critical_values'][0.99],
        'B': B,
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_path = OUTPUT_DIR / 'bootstrap_summary.csv'
summary_df.to_csv(summary_path, index=False)

print(f"\nExported: {summary_path}")
print(summary_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (scenario_name, data) in enumerate(results.items()):
    ax = axes[idx]
    T_boot = data['T_boot']
    T_obs = data['T_obs']
    critical_values = data['critical_values']
    
    ax.hist(T_boot, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(T_obs, color='red', linestyle='--', linewidth=2, label=f'T_obs={T_obs:.3f}')
    
    for i, (level, cv) in enumerate(critical_values.items()):
        colors = ['green', 'orange', 'darkred']
        ax.axvline(cv, color=colors[i], linestyle=':', linewidth=1.5, alpha=0.7)
    
    ax.set_xlabel('T_boot')
    ax.set_ylabel('Frequency')
    ax.set_title(scenario_name)
    ax.legend(['T_obs', 'c90', 'c95', 'c99'], fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / 'bootstrap_T_obs.png'
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
print(f"Exported: {plot_path}")
plt.close()

In [ ]:
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "git_commit": _get_git_commit(),
    "analysis": "bootstrap_null_synthetic",
    "input_paths": [],
    "output_paths": [
        str(bootstrap_results_path),
        str(summary_path),
        str(plot_path),
    ],
    "method": METHOD_NAME,
    "method_params": {
        "B": B,
        "p_values": P_VALUES,
        "critical_levels": CRITICAL_LEVELS,
        "p0": 1,
        "block_length": BLOCK_LENGTH,
    },
    "scenarios": [s[0] for s in SCENARIOS],
    "p_values": [float(s['p_value']) for _, s in results.items()],
    "random_seed": SEED,
    "software_versions": {
        "python": f"{sys.version.split()[0]}",
        "numpy": f"{np.__version__}",
        "pandas": f"{pd.__version__}",
        "matplotlib": f"{plt.matplotlib.__version__}",
    },
}

manifest_path = OUTPUT_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Exported: {manifest_path}")
print(json.dumps(manifest, indent=2))

In [ ]:
print("\nVerification:")
expected_files = [
    bootstrap_results_path,
    summary_path,
    plot_path,
    manifest_path,
]

all_exist = True
for fpath in expected_files:
    exists = fpath.exists()
    status = '✓' if exists else '✗'
    print(f"{status} {fpath.name} (size: {fpath.stat().st_size if exists else 'N/A'} bytes)")
    all_exist = all_exist and exists

if all_exist:
    print("\n✓ All outputs verified successfully")
else:
    print("\n✗ Some outputs are missing")